# 01. 의료영상 포맷 다루기 — DICOM·NIfTI

의료영상은 일반 이미지와 다른 포맷을 씁니다.
- **DICOM**: 병원 장비의 표준 포맷. 영상 + 환자·촬영 메타데이터 포함 (보통 슬라이스 1장당 1파일)
- **NIfTI**: 연구·딥러닝에서 널리 쓰는 3D 볼륨 포맷 (여러 슬라이스를 하나의 볼륨으로)

이 노트북에서는 두 포맷을 읽고 시각화합니다.

1. DICOM 읽기 (pydicom)
2. NIfTI 3D 볼륨 만들기·읽기 (nibabel)
3. SimpleITK로 볼륨 다루기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pydicom
import nibabel as nib
import SimpleITK as sitk

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
print('pydicom', pydicom.__version__, '| nibabel', nib.__version__)

## 1. DICOM 읽기

pydicom에 내장된 예제 CT 영상을 읽습니다. DICOM 파일에는 픽셀 데이터와 함께 환자·촬영 정보가 담겨 있습니다.
(실습에서는 `pydicom.dcmread('파일.dcm')`로 직접 불러옵니다.)

In [ ]:
from pydicom.data import get_testdata_file
ds = pydicom.dcmread(get_testdata_file('CT_small.dcm'))

print('Modality:', ds.Modality)
print('영상 크기:', ds.Rows, 'x', ds.Columns)
print('픽셀 배열 shape:', ds.pixel_array.shape)

plt.figure(figsize=(5, 5))
plt.imshow(ds.pixel_array, cmap='gray')
plt.title('DICOM CT 영상'); plt.axis('off')
plt.show()

## 2. NIfTI 3D 볼륨

3D 볼륨을 만들어 NIfTI로 저장하고 다시 읽습니다. 실제로는 CT/MRI 스캔이 이런 3D 볼륨으로 저장됩니다.
여기서는 구(sphere) 형태의 합성 볼륨을 만듭니다.

In [ ]:
# 합성 3D 볼륨 (구)
size = 64
zz, yy, xx = np.mgrid[:size, :size, :size]
center = size // 2
volume = ((xx-center)**2 + (yy-center)**2 + (zz-center)**2 < (size//3)**2).astype('float32')
volume += 0.1 * np.random.randn(*volume.shape)   # 잡음

# NIfTI로 저장하고 다시 읽기
nii = nib.Nifti1Image(volume, affine=np.eye(4))
nib.save(nii, '/workspace/sample_volume.nii.gz')
loaded = nib.load('/workspace/sample_volume.nii.gz').get_fdata()
print('볼륨 shape:', loaded.shape)

# 세 방향 단면 시각화
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(loaded[size//2, :, :], cmap='gray'); axes[0].set_title('axial')
axes[1].imshow(loaded[:, size//2, :], cmap='gray'); axes[1].set_title('coronal')
axes[2].imshow(loaded[:, :, size//2], cmap='gray'); axes[2].set_title('sagittal')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 3. SimpleITK로 볼륨 다루기

SimpleITK는 의료영상 I/O와 처리(리샘플링, 등록 등)에 강력한 라이브러리입니다. NIfTI를 읽어 기본 정보를 확인합니다.

In [ ]:
img = sitk.ReadImage('/workspace/sample_volume.nii.gz')
print('크기:', img.GetSize())
print('스페이싱:', img.GetSpacing())
arr = sitk.GetArrayFromImage(img)   # (z, y, x) numpy 배열
print('numpy 변환 shape:', arr.shape)

### 정리

- DICOM(pydicom)과 NIfTI(nibabel)로 의료영상을 읽고, 3D 볼륨을 단면으로 시각화했습니다.
- SimpleITK로 볼륨의 메타데이터(크기·스페이싱)를 확인하고 numpy로 변환했습니다.
- 다음 노트북에서는 MONAI transforms로 의료영상을 전처리·증강합니다.